# 第 23 课：流式 Beam/WFST 状态、Lattice 与热词

目标：理解 chunk 结束并不意味着句子结束；解码器必须保存搜索状态，并能输出候选 lattice/N-best。

<!-- course-upgrade-v2 -->
## 学习导航

| 项目 | 内容 |
|---|---|
| 所属阶段 | 语言模型与 WFST |
| 建议投入 | 3～5 小时，可分 2～3 次完成 |
| 前置要求 | 完成第 22 课；如果前测低于 2/3，先回看上一课小结 |
| 本课核心 | active state、lattice、stable prefix |
| 完成标准 | 能口头解释核心概念；独立完成强化题；从空白重写核心函数 |

高效顺序：**先回答前测 → 预测代码结果 → 再运行 → 修改一个变量 → 关闭答案复现 → 次日回忆。**


<!-- course-upgrade-v2 -->
## 课前诊断（先不要运行代码）

1. 分别用一句话解释：active state、lattice、stable prefix。
2. 画出这三个概念之间的输入—输出关系。
3. 写下你最不确定的一点，并给出一个暂时猜测。

自评：答对 0～1 题先复习前置课；答对 2 题可以正常学习；3 题都能讲清楚则直接挑战代码和迁移题。


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

def find_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "pyproject.toml").exists(): return p
    raise FileNotFoundError("请从 learn_asr 或 notebooks 目录启动 Jupyter")

ROOT = find_root()
BLANK = "∅"
plt.rcParams["figure.figsize"] = (11, 4)
print("项目根目录:", ROOT)

from collections import defaultdict
symbols=[BLANK,"A","B"]
P=np.array([[.6,.15,.55,.15,.6,.2],[.3,.7,.25,.15,.2,.1],[.1,.15,.2,.7,.2,.7]])

## 1. 跨 chunk 保存 prefix beam

In [ ]:
class StreamingPrefixBeam:
    def __init__(self,beam_size=5): self.beam={"":(1.,0.)};self.beam_size=beam_size
    def accept(self,Pchunk):
        for t in range(Pchunk.shape[1]):
            nxt=defaultdict(lambda:[0.,0.])
            for pref,(pb,pnb) in self.beam.items():
                for i,c in enumerate(symbols):
                    p=Pchunk[i,t]
                    if c==BLANK:nxt[pref][0]+=(pb+pnb)*p
                    elif pref and c==pref[-1]:
                        nxt[pref][1]+=pnb*p;nxt[pref+c][1]+=pb*p
                    else:nxt[pref+c][1]+=(pb+pnb)*p
            self.beam=dict(sorted(nxt.items(),key=lambda kv:sum(kv[1]),reverse=True)[:self.beam_size])
        return sorted(self.beam.items(),key=lambda kv:sum(kv[1]),reverse=True)

d=StreamingPrefixBeam()
for i in range(0,P.shape[1],2): print("chunk",i//2+1,"best",d.accept(P[:,i:i+2])[0])

如果每个 chunk 都重新从空前缀开始，重复字符、LM history 和候选路径全部丢失。WFST decoder 同样要保存 active states/tokens 及其代价。

## 2. Lattice 不只是 N-best 列表

Lattice 是许多共享前后缀候选的紧凑图，可以用于：

- best path；
- N-best；
- 置信度；
- 更大语言模型 rescoring；
- 时间对齐。

In [ ]:
candidates=[("AB",1.2,0.6),("AAB",1.4,0.4),("ABB",1.6,0.7),("BA",2.3,0.5)] # text, acoustic cost, lm cost
for scale in [0,.5,1,2]:
    ranked=sorted((a+scale*l,t) for t,a,l in candidates)
    print("LM scale",scale,"best",ranked[0])

## 3. Stable prefix 与 PGS

可以比较当前 top-K 假设的最长公共前缀，将共同部分标为 stable，其余作为可修改 partial。阈值越保守，字幕越稳但延迟越大。

In [ ]:
def common_prefix(strings):
    if not strings:return ""
    out=[]
    for chars in zip(*strings):
        if len(set(chars))==1:out.append(chars[0])
        else:break
    return "".join(out)
for hyps in [["北京天气","北京天启","北京天"],["今天天气","今天田气","今天"]]: print(hyps,"stable=",common_prefix(hyps))

## 4. Hotword 在 WFST 中的常见位置

可以动态组合小 grammar、调整指定路径权重，或在 beam search 中加 context score。关键是支持动态更新，同时限制误触发和图膨胀。

## 本课测试

1. chunk 结束后为何不能清空 beam？
2. lattice 与单个 best path 有何区别？
3. stable prefix 越长是否一定越好？
4. LM rescoring 为什么需要保留候选？
5. 动态热词为何不宜每次重编译整个巨大图？

<details><summary>展开参考答案</summary>

1. 需要保存前缀、重复状态和 LM/WFST 状态。2. lattice 紧凑保存多条候选。3. 不一定，过早稳定可能锁死错误。4. 被剪掉只剩一条后就无法翻盘。5. 成本和延迟高，应使用动态组合或上下文图。

</details>

<!-- course-upgrade-v2 -->
## 强化练习：第 23 课专属题库

请先把答案写进新的 Markdown/Code cell，再展开自评标准。

### A. 基础回忆

1. 不看上文，分别定义 `active state`、`lattice`、`stable prefix`。
2. 哪一个量/状态是本课最容易在模块边界丢失的？它的单位和 shape 是什么？
3. 本课至少写出两个“看起来能运行，但结果其实错误”的例子。

### B. 预测与推理

4. 场景：**chunk 结束时清空 decoder state**。先预测现象，再说明原因，最后给出一项可以验证猜测的指标。
5. 改变本课最关键参数的 0.5×、1×、2×，分别预测准确率、延迟、内存或数值误差怎样变化。
6. 画一张最小数据流图，在每条边标出 dtype、shape、时间单位或概率/代价方向。

### C. 编程与排错

7. 编程任务：**验证分块与整段 beam 最佳结果一致**。至少加入正常、边界、错误输入三类测试。
8. 故意制造一个 off-by-one、shape、状态未 reset 或数值稳定性错误；记录错误现象和定位过程。
9. 不看本课实现，从空白 cell 重写最核心函数，并用原实现作数值对照。

### D. 迁移与表达

10. 跨课任务：**连接 lattice rescoring、PGS 与热词**。
11. 用 90 秒向没有学过 ASR 的人解释本课；禁止只念术语，必须举一个数字或生活例子。
12. 写出一个生产系统中会监控的指标，以及它异常时优先检查的三处位置。

<details><summary>展开自评标准</summary>

- 每题 0～2 分：0=无法回答；1=方向正确但缺少单位、边界或验证；2=解释完整且能用代码/数字验证。
- 24 分满分：达到 19 分再进入下一课；15～18 分次日重做错题；低于 15 分回看本课图和核心代码。
- 第 4 题必须包含“预测—原因—指标”，第 7～9 题必须真正运行测试，第 10 题必须明确上下游 contract。
- 核心答案至少应正确使用：active state、lattice、stable prefix。

</details>


<!-- course-upgrade-v2 -->
## 间隔复习与离场票

### 离场票（现在完成）

- [ ] 我能不用笔记解释 active state、lattice、stable prefix。
- [ ] 我能说出本课最常见的错误及其观测现象。
- [ ] 我能从空白重写一个核心函数，并通过至少 3 个测试。
- [ ] 我能说明本课对上一层和下一层接口的影响。

### 复习时间表

- **明天（5 分钟）**：闭卷写出三个核心概念和一个公式/shape。
- **7 天后（15 分钟）**：重做第 4、7、10 题，不运行原答案。
- **30 天后（20 分钟）**：从真实音频或随机张量重新构造一个最小实验。

把错题记录到根目录 `LEARNING_LOG.md`。不要只写“不会”，要写：原判断、证据、正确规则、下次检查动作。
